In [0]:
%pip install --upgrade databricks-sdk==0.70.0
%restart_python

In [0]:
from databricks.sdk.service.jobs import JobSettings as Job

In [0]:
# Upgrade Databricks SDK to the latest version and restart Python to see updated packages
Project_bronze = Job.from_dict(
    {
        "name": "Project_bronze",
        "tasks": [
            {
                "task_key": "Parameters",
                "spark_python_task": {
                    "python_file": "/Workspace/extra/parameters.py",
                },
                "min_retry_interval_millis": 15000,
                "disable_auto_optimization": False,
                "environment_key": "Default",
            },
            {
                "task_key": "Bronze_Autoloader",
                "depends_on": [
                    {
                        "task_key": "Parameters",
                    },
                ],
                "for_each_task": {
                    "inputs": "{{tasks.Parameters.values.output_datasets}}",
                    "task": {
                        "task_key": "Bronze_Autoloader_iteration",
                        "notebook_task": {
                            "notebook_path": "/Workspace/extra/Bronze_autoloader",
                            "base_parameters": {
                                "file_name": "{{input.file_name}}",
                            },
                            "source": "WORKSPACE",
                        },
                    },
                },
            },
        ],
        "queue": {
            "enabled": True,
        },
        "environments": [
            {
                "environment_key": "Default",
                "spec": {
                    "environment_version": "4",
                },
            },
        ],
        "performance_target": "PERFORMANCE_OPTIMIZED",
    }
)

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
w.jobs.reset(new_settings=Project_bronze, job_id=977998239018927)
# or create a new job using: w.jobs.create(**Project_bronze.as_shallow_dict())
